# Train PDP_IOD trên RPC (Kaggle)

**Trước khi chạy** (cột bên phải):
1. *Add Input* → thêm 2 dataset: `pdp-iod-code` và `rpc-checkout-data`.
2. *Session options* → **Accelerator: GPU T4 x2**, **Internet: On** (cần để cài thư viện và tải model `SenseTime/deformable-detr`).
3. Chạy **lần lượt từng ô từ trên xuống**, đợi ô trước xong mới chạy ô sau. Không dùng `&`/`nohup` trong notebook này.

## Các chế độ (`MODE` ở ô 1)
| MODE | Làm gì | Thời gian | Cách đọc kết quả |
|---|---|---|---|
| `'A'` | Task 1, `group_mini`, 10 epoch, **cập nhật mỗi 2 ảnh** (≈2500 bước), đánh giá trên **test** | ~25 phút | AP Task 1 ≥ ~0.05 → trước đó chỉ thiếu bước huấn luyện, chạy `'full'` |
| `'B'` | Giống A nhưng đánh giá trên **245 ảnh thật của tập train** | ~25 phút | AP cao ở B mà thấp ở A → vấn đề tổng quát hoá (ảnh ghép / vật không gán nhãn); AP thấp cả hai → mô hình không học nổi (thử `EXTRA_ARGS`) |
| `'full'` | Chạy chính 4 task trên `group` (ảnh thật + ghép) | nhiều giờ | *Save Version → Save & Run All* (nền, tối đa 12 giờ) |

Chạy A trước. Chỉ chạy B nếu AP của A vẫn ≈ 0.00x. Đổi `MODE` → chạy lại từ ô 1 (ô 3 không cần chạy lại nếu session còn sống).

Chạy tiếp từ lần trước: đặt `START_TASK` = task chưa xong, *Add Input* chính notebook cũ (bản đã chạy xong). Giữ nguyên `EPOCHS`.

Ô 3 cài thư viện (lần đầu của mỗi session) rồi kiểm tra trong tiến trình riêng; cuối ô phải in `import model OK`. **Không cần restart kernel.**

In [1]:
# ===== 1. CẤU HÌNH =====
MODE = 'A'         # 'A' = thử nhiều bước hơn | 'B' = đánh giá trên ảnh train thật | 'full' = chạy chính

if MODE == 'A':
    SPLIT, EXP, EPOCHS, N_TASKS, EXTRA_ARGS = 'group_mini', 'thu_acc2', 10, 1, ''
elif MODE == 'B':
    SPLIT, EXP, EPOCHS, N_TASKS, EXTRA_ARGS = 'group_mini', 'thu_ovf', 10, 1, ''
    # nếu A và B đều AP ~0: thử mở decoder: EXTRA_ARGS = '--freeze backbone,encoder --lr 2e-4'
else:
    SPLIT, EXP, EPOCHS, N_TASKS, EXTRA_ARGS = 'group', 'rpc_group', 8, 4, ''
    # 'group' = ảnh thật + ảnh ghép | 'group_real' = chỉ ảnh thật (ablation) -> đổi EXP cho khác nhau

# Số ảnh mỗi lần cập nhật trọng số (code gốc: 32). Mini chỉ có 500 ảnh -> 2 để có ~250 bước/epoch. Chạy full: giữ code gốc.
ACC_TARGET = 2 if MODE in ('A', 'B') else None

START_TASK = 1     # chạy tiếp: đặt = task chưa xong (vd. 3), nhớ Add Input notebook cũ
IMG_SIZE   = 512   # bài báo dùng 800
N_GPUS     = 1     # thử 1 trước cho chắc; 2 nếu muốn dùng cả T4 x2
BATCH      = 2
ENV_EXTRA  = ''    # biến cho run_rpc.sh, vd. 'PSEUDO_HIGH=0.4 PSEUDO_LOW=0.15'
REUSE_TASK1_FROM = ''   # dùng lại Task_1 của thí nghiệm khác (vd. 'rpc_group_E1') rồi đặt START_TASK = 2
print(dict(MODE=MODE, SPLIT=SPLIT, EXP=EXP, EPOCHS=EPOCHS, N_TASKS=N_TASKS, EXTRA_ARGS=EXTRA_ARGS, ACC_TARGET=ACC_TARGET))

{'MODE': 'A', 'SPLIT': 'group_mini', 'EXP': 'thu_acc2', 'EPOCHS': 10, 'N_TASKS': 1, 'EXTRA_ARGS': '', 'ACC_TARGET': 2}


In [2]:
# ===== 2. TÌM CODE + DỮ LIỆU (nhanh, không quét hết ảnh) =====
import glob, os, shutil, zipfile
W = '/kaggle/working'
SKIP = {'val2019','test2019','train2019','synth','images','cutouts','backgrounds','instances'}

def walk(base, maxdepth=6):
    base = base.rstrip('/')
    for d, dirs, files in os.walk(base):
        depth = d[len(base):].count('/')
        dirs[:] = [x for x in dirs if x not in SKIP] if depth < maxdepth else []
        yield d, dirs, files

# code: thư mục có cả run_rpc.sh và main.py (ưu tiên dataset pdp-iod-code)
code_dir = [d for d, _, f in walk('/kaggle/input') if 'run_rpc.sh' in f and 'main.py' in f]
code_dir = sorted(code_dir, key=lambda p: 'pdp-iod-code' not in p)
assert code_dir, 'Chưa Add Input dataset pdp-iod-code'
shutil.rmtree(f'{W}/PDP_IOD', ignore_errors=True)
shutil.copytree(code_dir[0], f'{W}/PDP_IOD')     # /kaggle/input chỉ đọc -> chép sang chỗ ghi được

# dữ liệu: thư mục chứa tasks_pdp + test2019
roots = [os.path.dirname(os.path.join(d, x)) for d, dirs, _ in walk('/kaggle/input', 4)
         for x in dirs if x == 'tasks_pdp' and os.path.isdir(os.path.join(d, 'test2019'))]
if not roots:                                    # dataset là file zip -> giải nén
    zips = glob.glob('/kaggle/input/**/archive.zip', recursive=True)
    assert zips, 'Chưa Add Input dataset rpc-checkout-data'
    print('Giải nén', zips[0], '-> /kaggle/tmp (vài phút)...')
    os.makedirs('/kaggle/tmp', exist_ok=True)
    zipfile.ZipFile(zips[0]).extractall('/kaggle/tmp/archive')
    roots = [os.path.join(d) for d, dirs, _ in walk('/kaggle/tmp/archive', 4) if 'tasks_pdp' in dirs]
ROOT = roots[0]
OUT = f'{W}/runs'
assert os.path.isdir(f'{ROOT}/tasks_pdp/{SPLIT}'), f'Không có {ROOT}/tasks_pdp/{SPLIT}: {os.listdir(ROOT + "/tasks_pdp")}'
print('Code :', code_dir[0]); print('ROOT :', ROOT)
print('Có', SPLIT, ':', sorted(os.listdir(f'{ROOT}/tasks_pdp/{SPLIT}'))[:8])

# ----- vá code (ô này chép lại code gốc mỗi lần, nên bản vá được áp dụng lại ở đây) -----
def patch(fname, old, new):
    p = f'{W}/PDP_IOD/{fname}'; s = open(p).read()
    assert s.count(old) == 1, f'{fname}: không tìm thấy đúng 1 chỗ cần sửa (code đã đổi?)'
    open(p, 'w').write(s.replace(old, new)); print('đã vá', fname)
if ACC_TARGET:
    patch('main.py', 'int(32/(args.n_gpus*args.batch_size))', f'int({ACC_TARGET}/(args.n_gpus*args.batch_size))')

# MODE B: đánh giá trên ảnh THẬT của tập train. ROOT gốc chỉ đọc -> tạo bản "overlay" (symlink) có tasks_pdp ghi được.
if MODE == 'B':
    import json
    SRC, OV, D = ROOT, f'{W}/rootov', 'group_mini_ovf'
    shutil.rmtree(OV, ignore_errors=True); os.makedirs(f'{OV}/tasks_pdp')
    for n in os.listdir(SRC):
        if n != 'tasks_pdp': os.symlink(f'{SRC}/{n}', f'{OV}/{n}')
    shutil.copytree(f'{SRC}/tasks_pdp/{SPLIT}', f'{OV}/tasks_pdp/{D}')
    p = f'{OV}/tasks_pdp/{D}'
    tr = json.load(open(f'{p}/train_task_1.json'))
    real = [i for i in tr['images'] if i['file_name'].startswith('val2019')]
    ids = {i['id'] for i in real}
    te = dict(tr)
    te['images'] = [dict(i, file_name='../' + i['file_name']) for i in real]   # test_img_dir = ROOT/test2019 -> ../val2019/...
    te['annotations'] = [a for a in tr['annotations'] if a['image_id'] in ids]
    json.dump(te, open(f'{p}/test_task_1.json', 'w'))
    print('MODE B: đánh giá trên', len(te['images']), 'ảnh thật của tập train |', len(te['annotations']), 'box')
    ROOT, SPLIT = OV, D

# chạy tiếp: lấy checkpoint của lần chạy trước (notebook cũ được Add Input)
def find_runs(name):
    return [os.path.join(d, name) for d, dirs, _ in walk('/kaggle/input', 6)
            if os.path.basename(d) == 'runs' and name in dirs]
old = find_runs(EXP)
if old and START_TASK > 1:
    shutil.copytree(old[0], f'{OUT}/{EXP}', dirs_exist_ok=True)
    print('Đã chép kết quả cũ từ', old[0], ':', sorted(os.listdir(f'{OUT}/{EXP}')))
elif REUSE_TASK1_FROM and START_TASK == 2:
    src = [p + '/Task_1' for p in find_runs(REUSE_TASK1_FROM)]
    assert src, f'Không thấy runs/{REUSE_TASK1_FROM}/Task_1 trong Input'
    shutil.copytree(src[0], f'{OUT}/{EXP}/Task_1', dirs_exist_ok=True)
    print('Dùng lại Task_1 từ', src[0])
elif START_TASK > 1:
    print('CẢNH BÁO: START_TASK > 1 nhưng không thấy kết quả cũ runs/%s trong Input' % EXP)

Code : /kaggle/input/datasets/hinvminh12/pdp-iod-code
ROOT : /kaggle/input/datasets/hinvminh12/rpc-checkout-data
Có group_mini : ['rpc_task_info.json', 'test_task_1.json', 'test_task_12.json', 'test_task_123.json', 'test_task_1234.json', 'test_task_2.json', 'test_task_3.json', 'test_task_4.json']
đã vá main.py


In [4]:
# ===== 3. CÀI THƯ VIỆN (dùng đúng requirements.txt của code) =====
# Các dòng "dependency conflicts" màu đỏ khi cài là cảnh báo của gói Kaggle có sẵn, bỏ qua được.
from importlib.metadata import version, PackageNotFoundError
def ver(p):
    try: return version(p)
    except PackageNotFoundError: return None
if not (ver('transformers') == '4.37.2' and (ver('numpy') or '').startswith('1.')):
    !cd /kaggle/working/PDP_IOD && pip install -q -r requirements.txt
# jax/flax/tensorflow của Kaggle đòi numpy 2 -> làm transformers 4.37.2 import lỗi; đồ án không dùng. ninja: tránh biên dịch kernel CUDA.
!pip uninstall -y -q jax jaxlib flax tensorflow ninja 2>/dev/null; true

# Kiểm tra trong một tiến trình MỚI (giống cách ô 4 chạy), không import vào kernel notebook -> không cần restart
!USE_TF=0 USE_JAX=0 python -c "import numpy, torch, transformers; print('numpy', numpy.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__); print('GPU:', torch.cuda.device_count(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'); from transformers import DeformableDetrForObjectDetection; print('import model OK')"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 94.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 93.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.7/777.7 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 29.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.8 MB/s eta

In [ ]:
# ===== 4. TRAIN (task START_TASK -> N_TASKS, nối tiếp) =====
# Log hiện trực tiếp ở dưới ô này (qua tee) và lưu vào /kaggle/working/train_<EXP>.log
env = f'ROOT="{ROOT}" OUT="{OUT}" SPLIT={SPLIT} EXP={EXP} EPOCHS={EPOCHS} START_TASK={START_TASK} N_TASKS={N_TASKS} ' \
      f'IMG_SIZE={IMG_SIZE} N_GPUS={N_GPUS} BATCH={BATCH} WORKERS=4 ACCEL=gpu PYTHON=python USE_TF=0 USE_JAX=0 ' + ENV_EXTRA
!cd /kaggle/working/PDP_IOD && {env} bash run_rpc.sh {EXTRA_ARGS} 2>&1 | tee -a /kaggle/working/train_{EXP}.log

train_task_1.json         500 ảnh     4077 box  thiếu ảnh: 0  sai lớp: 0
train_task_2.json         500 ảnh     3845 box  thiếu ảnh: 0  sai lớp: 0
train_task_3.json         500 ảnh     3895 box  thiếu ảnh: 0  sai lớp: 0
train_task_4.json         500 ảnh     3777 box  thiếu ảnh: 0  sai lớp: 0
test_task_1.json          779 ảnh     2954 box  thiếu ảnh: 0  sai lớp: 0
test_task_2.json          876 ảnh     4119 box  thiếu ảnh: 0  sai lớp: 0
test_task_3.json          806 ảnh     2994 box  thiếu ảnh: 0  sai lớp: 0
test_task_4.json          685 ảnh     2077 box  thiếu ảnh: 0  sai lớp: 0
test_task_12.json         979 ảnh     7073 box  thiếu ảnh: 0  sai lớp: 0
test_task_123.json        996 ảnh    10067 box  thiếu ảnh: 0  sai lớp: 0
test_task_1234.json      1000 ảnh    12144 box  thiếu ảnh: 0  sai lớp: 0
OK, dữ liệu đầy đủ
Training thu_acc2: task 1..1, 10 epoch/task -> /kaggle/working/runs/thu_acc2
Seed set to 42
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:1142: FutureW

In [ ]:
# ===== 5. KẾT QUẢ =====
for f in sorted(glob.glob(f'{OUT}/{EXP}/Task_*/stats.txt')) + sorted(glob.glob(f'{OUT}/{EXP}/final_stats.txt')):
    print('=====', f); print(open(f).read())
for f in glob.glob(f'{OUT}/{EXP}/**/*.ckpt', recursive=True):   # .ckpt của Lightning trùng với checkpointXX.pth -> xoá cho nhẹ Output
    os.remove(f)
shutil.rmtree(f'{W}/PDP_IOD', ignore_errors=True)               # không lưu bản sao code vào Output
!du -sh {OUT}